In [ ]:
import os
import streamlit as st
import pickle
import time
import langchain
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.document_loaders import UnstructuredURLLoader
from langchain_classic.embeddings import OpenAIEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_openai import ChatOpenAI


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
from langchain_community.llms import OpenAI
# Initialise LLM with required params
llm = OpenAI(temperature=0.9, max_tokens=500) 

### (1) Load data

In [ ]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

### (2) Split data to create chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)

In [ ]:
len(docs)

In [ ]:
docs[0]

### (3) Create embeddings for these chunks and save them to FAISS index

In [ ]:
# Create the embeddings of the chunks using openAIEmbeddings
embeddings = OpenAIEmbeddings()

# Pass the documents and embeddings inorder to create FAISS vector index
vectorindex_openai = FAISS.from_documents(docs, embeddings)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Storing vector index create in local
vectorindex_openai = FAISS.from_documents(docs, embeddings)
vectorindex_openai.save_local("faiss_index")

In [ ]:
if os.path.exists("faiss_index"):
    vectorIndex = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

### (4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer

In [ ]:
chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vectorIndex.as_retriever())
chain

In [ ]:
query = "what is the price of Tiago iCNG?"
# query = "what are the main features of punch iCNG?"

langchain.debug=True

chain({"question": query}, return_only_outputs=True)